## Download FAIR1M from HuggingFace

In [1]:
# from huggingface_hub import snapshot_download

# # Download the dataset files to a local directory named "FAIR1M"
# snapshot_download(
#     repo_id="blanchon/FAIR1M",
#     repo_type="dataset",
#     local_dir="FAIR1M",
#     local_dir_use_symlinks=False,
# )

# print("Dataset downloaded to the 'FAIR1M' directory.")

## Convert Dataset to COCO Format

In [2]:
import os
import json
import xml.etree.ElementTree as ET
import numpy as np
from tqdm import tqdm
import datetime

# --- Configuration ---
# Adjust these paths to match your folder structure
ANNOTATIONS_DIR = '/caa/Homes01/mburges/datasets/FAIR1M/labelXmls'
IMAGES_DIR = '/caa/Homes01/mburges/datasets/FAIR1M/images'
OUTPUT_JSON_PATH = '/caa/Homes01/mburges/datasets/FAIR1M/coco_annotations.json'
# ---------------------

def get_polygon_area(points):
    """Calculates the area of a polygon using the Shoelace formula."""
    x = points[:, 0]
    y = points[:, 1]
    return 0.5 * np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))

def main():
    coco_output = {
        "info": {
            "description": "Converted FAIR1M Dataset",
            "version": "1.0",
            "year": datetime.date.today().year,
            "date_created": datetime.datetime.now().isoformat(' ')
        },
        "licenses": [],
        "images": [],
        "annotations": [],
        "categories": []
    }

    category_map = {}
    category_id_counter = 1
    annotation_id_counter = 1
    image_id_counter = 1
    
    # Get all XML files from the annotations directory
    xml_files = [f for f in os.listdir(ANNOTATIONS_DIR) if f.endswith('.xml')]
    
    print(f"Found {len(xml_files)} XML files. Starting conversion...")

    for xml_file in tqdm(xml_files, desc="Processing XMLs"):
        xml_path = os.path.join(ANNOTATIONS_DIR, xml_file)
        
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()

            # --- Image Information ---
            filename = root.find('source/filename').text
            width = int(root.find('size/width').text)
            height = int(root.find('size/height').text)

            image_info = {
                "id": image_id_counter,
                "file_name": filename,
                "width": width,
                "height": height,
                "license": None,
                "date_captured": None
            }
            coco_output["images"].append(image_info)

            # --- Annotations ---
            for obj in root.findall('objects/object'):
                category_name = obj.find('possibleresult/name').text
                
                # --- Categories ---
                if category_name not in category_map:
                    category_map[category_name] = category_id_counter
                    category_info = {
                        "id": category_id_counter,
                        "name": category_name,
                        "supercategory": "object" # You can customize this
                    }
                    coco_output["categories"].append(category_info)
                    category_id_counter += 1
                
                category_id = category_map[category_name]
                
                # --- Segmentation and BBox ---
                points_str = [p.text for p in obj.findall('points/point')]
                # The XML has 5 points (last one repeats first), take the first 4 for the quad
                points = np.array([list(map(float, p.split(','))) for p in points_str[:-1]])
                
                # Flatten for segmentation field: [x1, y1, x2, y2, ...]
                segmentation = [points.flatten().tolist()]
                
                # Calculate area
                area = get_polygon_area(points)

                # Calculate axis-aligned bounding box [x_min, y_min, width, height]
                x_min = float(np.min(points[:, 0]))
                y_min = float(np.min(points[:, 1]))
                x_max = float(np.max(points[:, 0]))
                y_max = float(np.max(points[:, 1]))
                bbox_width = x_max - x_min
                bbox_height = y_max - y_min
                
                annotation_info = {
                    "id": annotation_id_counter,
                    "image_id": image_id_counter,
                    "category_id": category_id,
                    "segmentation": segmentation,
                    "area": float(area),
                    "bbox": [x_min, y_min, bbox_width, bbox_height],
                    "iscrowd": 0
                }
                coco_output["annotations"].append(annotation_info)
                annotation_id_counter += 1
            
            image_id_counter += 1

        except Exception as e:
            print(f"\nError processing file {xml_file}: {e}")

    # --- Save the JSON file ---
    with open(OUTPUT_JSON_PATH, 'w') as f:
        json.dump(coco_output, f, indent=4)
        
    print(f"\nConversion complete! 🎉")
    print(f"Total images: {len(coco_output['images'])}")
    print(f"Total annotations: {len(coco_output['annotations'])}")
    print(f"Total categories ({len(coco_output['categories'])}): {list(category_map.keys())}")
    print(f"COCO JSON file saved to: {OUTPUT_JSON_PATH}")


if __name__ == "__main__":
    main()

Found 1732 XML files. Starting conversion...


Processing XMLs: 100%|██████████| 1732/1732 [00:04<00:00, 360.33it/s]



Conversion complete! 🎉
Total images: 1732
Total annotations: 82938
Total categories (37): ['Liquid Cargo Ship', 'Passenger Ship', 'Cargo Truck', 'Small Car', 'Dump Truck', 'Van', 'Dry Cargo Ship', 'Excavator', 'Intersection', 'other-vehicle', 'Boeing737', 'A321', 'Tennis Court', 'Basketball Court', 'A220', 'Motorboat', 'Fishing Boat', 'other-airplane', 'Boeing787', 'Engineering Ship', 'Warship', 'Tugboat', 'other-ship', 'Tractor', 'Bus', 'Roundabout', 'ARJ21', 'Boeing747', 'Football Field', 'Trailer', 'Truck Tractor', 'Bridge', 'Baseball Field', 'A330', 'A350', 'Boeing777', 'C919']
COCO JSON file saved to: /caa/Homes01/mburges/datasets/FAIR1M/coco_annotations.json
